In [1]:
import numpy as np
import pandas as pd
import nibabel as nib
from nilearn.image import new_img_like


In [2]:
atlas_nii = nib.load('/media/storage/MATLAB_atlases-20250612T134418Z-1-001/MATLAB_atlases/schaefer2018tian2020_400_7.nii')
atlas_data = atlas_nii.get_fdata()

atlas_csv = pd.read_csv('/home/gaia/Projects/legacy_data/legacy_pipe/models/atlases/space-MNI152_atlas-schaefer2018tian2020_res-1mm_den-400_div-7networks_dseg.csv')
coef_df = pd.read_csv('/home/gaia/Projects/legacy_data/legacy_pipe/data/processed/birth_year_coef_df_with_stability.csv')
coef_df = coef_df[coef_df['stable_95'] == True]

positive_coef_df = coef_df[coef_df['coef'] > 0]
negative_coef_df = coef_df[coef_df['coef'] < 0]

In [3]:
# Merge your results with the atlas metadata to keep everything aligned 
# This pairs your statistical effects with 'name', 'network', and 'hemisphere' 
merged_results = pd.merge(
    atlas_csv, 
    coef_df, 
    left_on='index', 
    right_on='index', 
    how='inner'
)

# Split into positive and negative dataframes for separate decoding pipelines
positive_effects = merged_results[merged_results['t'] > 0].copy()
negative_effects = merged_results[merged_results['t'] < 0].copy()

In [9]:
negative_rois = negative_effects['name'].tolist()
# remove duplicates 
negative_rois = list(set(negative_rois))
positive_rois = positive_effects['name'].tolist()
# remove duplicates 
positive_rois = list(set(positive_rois))

In [14]:
negative_subcortex = [roi for roi in negative_rois if '7Network' not in roi]
positive_subcortex = [roi for roi in positive_rois if '7Network' not in roi]

negative_cortex = [roi for roi in negative_rois if '7Network' in roi]
positive_cortex = [roi for roi in positive_rois if '7Network' in roi]

print("total Negative Subcortical ROIs:", len(negative_subcortex))
print("total Positive Subcortical ROIs:", len(positive_subcortex))

print("total Negative Cortical ROIs:", len(negative_cortex))
print("total Positive Cortical ROIs:", len(positive_cortex))

total Negative Subcortical ROIs: 14
total Positive Subcortical ROIs: 15
total Negative Cortical ROIs: 13
total Positive Cortical ROIs: 106


In [17]:
negative_somatomotor = [roi for roi in negative_cortex if 'SomMot' in roi]
positive_somatomotor = [roi for roi in positive_cortex if 'SomMot' in roi]

print("total Negative Somatomotor ROIs:", len(negative_somatomotor))
print("total Positive Somatomotor ROIs:", len(positive_somatomotor))

total Negative Somatomotor ROIs: 5
total Positive Somatomotor ROIs: 25


In [4]:
birth_year_coef_df = pd.read_csv("/home/gaia/Projects/legacy_data/legacy_pipe/data/processed/birth_year_coef_df_with_stability.csv")

birth_year_coef_df['sig_stab'] = (birth_year_coef_df['fdr_p'] < 0.05) & (birth_year_coef_df['stable_95'] == True)

In [5]:
# Mapping the subcortex

# # print a list of unique names from column name in atlas_csv, where network=='subcortex'
# subcortex_names = atlas_csv[atlas_csv['network'] == 'subcortex']['name'].unique()
# print(subcortex_names)

# tian_hcp_network_mapping = {
#     # --- RIGHT HEMISPHERE (_rh) ---
#     # Hippocampus
#     'HIP-head-m1-rh': 'default',
#     'HIP-head-m2-rh': 'default',
#     'HIP-head-l-rh': 'limbic',
#     'HIP-body-rh': 'limbic',
#     'HIP-tail-rh': 'default',
    
#     # Amygdala
#     'lAMY-rh': 'limbic',
#     'mAMY-rh': 'limbic',
    
#     # Nucleus Accumbens / Striatum
#     'NAc-shell-rh': 'limbic',
#     'NAc-core-rh': 'control',
#     'PUT-VA-rh': 'control',
#     'PUT-DA-rh': 'control',
#     'PUT-VP-rh': 'salience / ventral attention',
#     'PUT-DP-rh': 'somatomotor',
#     'CAU-VA-rh': 'default',
#     'CAU-DA-rh': 'control',
#     'CAU-body-rh': 'control',
#     'CAU-tail-rh': 'control',
    
#     # Globus Pallidus
#     'aGP-rh': 'limbic',
#     'pGP-rh': 'somatomotor',
    
#     # Thalamus
#     'THA-VAia-rh': 'control',
#     'THA-VAip-rh': 'control',
#     'THA-VAs-rh': 'salience / ventral attention',
#     'THA-DAm-rh': 'salience / ventral attention',
#     'THA-DAl-rh': 'dorsal attention',
#     'THA-VPm-rh': 'somatomotor',
#     'THA-VPl-rh': 'somatomotor',
#     'THA-DP-rh': 'visual',

#     # --- LEFT HEMISPHERE (_lh) ---
#     # Hippocampus
#     'HIP-head-m1-lh': 'default',
#     'HIP-head-m2-lh': 'default',
#     'HIP-head-l-lh': 'limbic',
#     'HIP-body-lh': 'limbic',
#     'HIP-tail-lh': 'default',
    
#     # Amygdala
#     'lAMY-lh': 'limbic',
#     'mAMY-lh': 'limbic',
    
#     # Nucleus Accumbens / Striatum
#     'NAc-shell-lh': 'limbic',
#     'NAc-core-lh': 'control',
#     'PUT-VA-lh': 'control',
#     'PUT-DA-lh': 'control',
#     'PUT-VP-lh': 'salience / ventral attention',
#     'PUT-DP-lh': 'somatomotor',
#     'CAU-VA-lh': 'default',
#     'CAU-DA-lh': 'control',
#     'CAU-body-lh': 'control',
#     'CAU-tail-lh': 'control',
    
#     # Globus Pallidus
#     'aGP-lh': 'limbic',
#     'pGP-lh': 'somatomotor',
    
#     # Thalamus
#     'THA-VAia-lh': 'control',
#     'THA-VAip-lh': 'control',
#     'THA-VAs-lh': 'salience / ventral attention',
#     'THA-DAm-lh': 'salience / ventral attention',
#     'THA-DAl-lh': 'dorsal attention',
#     'THA-VPm-lh': 'somatomotor',
#     'THA-VPl-lh': 'somatomotor',
#     'THA-DP-lh': 'visual'
# }

# # This will look at the 'roi_name' column, map it to the network, and overwrite 'subcortex'
# birth_year_coef_df['network'] = birth_year_coef_df.apply(
#     lambda row: tian_hcp_network_mapping.get(row['region_name'], row['network']) 
#     if row['network'] == 'subcortex' else row['network'], 
#     axis=1
# )

In [6]:
# 1. Group by age bin and network, then aggregate total and significant ROIs
summary_df = (
    birth_year_coef_df.groupby(['age_bin', 'network'])
    .agg(
        total_rois=('sig_stab', 'count'),
        sig_stab_rois=('sig_stab', 'sum')  # Summing a boolean column counts the True values
    )
    .reset_index()
)

# 2. Calculate the percentage
summary_df['percentage'] = (summary_df['sig_stab_rois'] / summary_df['total_rois']) * 100

# 3. Reorder and rename columns to match your exact requested format
columns_order = ['network', 'age_bin', 'total_rois', 'sig_stab_rois', 'percentage']
summary_df = summary_df[columns_order]

# Display the resulting table
print(summary_df)

                         network   age_bin  total_rois  sig_stab_rois  \
0                        control  (20, 25]          52              3   
1                        default  (20, 25]          91              7   
2               dorsal attention  (20, 25]          46              1   
3                         limbic  (20, 25]          26              0   
4   salience / ventral attention  (20, 25]          47             21   
5                    somatomotor  (20, 25]          77             19   
6                      subcortex  (20, 25]          54             10   
7                         visual  (20, 25]          61              0   
8                        control  (25, 30]          52              3   
9                        default  (25, 30]          91             14   
10              dorsal attention  (25, 30]          46              1   
11                        limbic  (25, 30]          26              4   
12  salience / ventral attention  (25, 30]         